# Congressional Trading Signal (STOCK Act / PTR) — research notebook

**Slug:** `congressional-trading-signal`  ·  **Date:** 2026-06-19  ·  **Researcher:** Kelvin  ·  **Status:** `reject`

**Decision record:** [`docs/research_decisions/2026-06-19_congressional_trading_signal.md`](../../docs/research_decisions/2026-06-19_congressional_trading_signal.md)  
**Full report (all figures + robustness):** [`reports/congress_signal_report.html`](../../reports/congress_signal_report.html) — regenerate with `PYTHONPATH=src python scripts/congress_signal_report.py`

> This notebook is intentionally thin: the reusable logic lives in `src/alpha_lab/` (`data/loaders/congress.py`, `data/congress_universe.py`, `backtest/congress_signal.py`, `backtest/congress_book.py`, `analytics/event_study.py`, `stats/tests.py`). It loads the package, reproduces the headline, and records the verdict.

## Research question

Is there a **tradable** signal in congressional STOCK-Act disclosures once single-stock flow is aggregated to GICS sectors and expressed through **ETFs only** (no individual names), net of costs + cost of cash (2018–2026), that beats SPY and the NANC/KRUZ copy-ETFs?

## Hypothesis & economic rationale

Persistent, multi-member, same-direction sector accumulation *might* be a medium-term sector-tilt signal. **Skeptic's prior (the one to beat):** the 45-day disclosure lag eats the sharp drift; post-STOCK-Act studies find the edge largely gone; NANC/KRUZ returns are tech *beta*, not selection. We try to **falsify**, with `filing_date` (public date) as the only valid signal date.

In [ ]:
import warnings, logging
warnings.filterwarnings('ignore')
for _n in ('httpx','yfinance','urllib3','peewee'): logging.getLogger(_n).setLevel(logging.CRITICAL)
import pandas as pd
from alpha_lab.backtest.congress_book import load_congress_book_data, run_study, latest_target_weights
from alpha_lab.backtest.metrics import summary

bd = load_congress_book_data()              # kadoa PTRs + curated sector map + ETF prices + rf
print('eval', bd.eval_start.date(), '| rf', bd.rf_source, '| flow mapped %.0f%%' % bd.coverage['pct_flow_mapped'])
s = run_study(bd)                           # Angle A: dollar-neutral sector net-flow L/S

In [ ]:
# Headline: strategy (excess of cash) vs the three benchmarks it must beat
rows = {'Congress sector L/S (excess of cash)': s['strategy_summary'], **{k: v for k, v in s['benchmark_summary'].items()}}
pd.DataFrame(rows).T[['Sharpe','CAGR','AnnVol','MaxDD','NPeriods']].round(3)

In [ ]:
# Equity curves (excess of cash) vs SPY + sector equal-weight
from alpha_lab.reporting.charts import equity_curve
eq = pd.DataFrame({'Congress sector L/S': s['excess_returns'], 'SPY': s['benchmarks']['SPY'],
                   'Sector EW': s['benchmarks']['SectorEW']}).dropna()
equity_curve(eq)

## Decision

**`reject` as a tradable ETF strategy.** A small, real, **buy-side-only, short-horizon** information signal exists (event study: filing-date buy CAR ≈ +0.4%/42d, t≈2.4; rank-IC +0.05@21d then reverses by 63d), but it does **not survive** translation to sector-ETF expression + beta-neutralization + costs: Sharpe ≈ 0.12 vs SPY 0.82 / NANC 1.30, bootstrap CI spans 0, **Deflated Sharpe 0.15**. The leg decomposition shows long-top-3 ≈ market beta and the dollar-neutral book ≈ 0 alpha — i.e. NANC's edge is beta, as the prior warned. Adversarially audited (quant-skeptic): trustworthy null.

**Next:** Angle B (committee-overlap weighting) is the one untested place a subset edge could hide — blocked on point-in-time committee rosters (scaffolded in `congress_book.committee_weighted_flow`). Otherwise: use the sector-flow z-score as a low-correlation *idea flag*, not a standalone book. Do **not** advance to paper/live. See the decision record + report for full robustness.

In [ ]:
# Execution handoff (research-only): most-recent sector-ETF target weights a future
# quant_bot_manager strategy could adapt. NOT wired to any live bot (verdict = reject).
latest_target_weights(bd).round(3)